# Fig. 2: Full model (cumulant expansion)

Steady state of the full cooling model (cavity mode $a$, auxiliary mode $b$ and $N_c$ effective two-level atoms) on a grid of squeezing parameter $r$ × number of cooling atoms $N_c$, using a second-order cumulant expansion (QuantumCumulants.jl). From it we obtain

* the quadrature fluctuations $\Delta\hat X$ (**panel a**),
* the squeezing spectrum $S^{\rm sq}_{\hat X}(\nu)$, its minimum over $\nu$ on the whole grid and a cut at fixed $r$ (**panel c**),
* the analytical effective-model prediction for $\Delta\hat X$ and the validity conditions of the adiabatic eliminations (dashed lines in panels a and c).

**Outputs** (`../data/`): `N_c.npy`, `r.npy`, `freqs_MHz.npy`, `dX_full.npy`, `dX_eff.npy`, `Ssq_min_full.npy`, `Ssq_cut_full.npy`, `cond_r.npy`, `cond_chi.npy`, `cond_g0.npy`.

The loop over $r$ is parallelised with `Threads.@threads`: start Julia with several threads (e.g. `julia -t auto`). The 150 × 150 grid is expensive.

In [ ]:
using QuantumCumulants
using ModelingToolkit
using OrdinaryDiffEq
using WignerSymbols
using ProgressMeter
using NPZ

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)

    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Grid
num_r = 150 # Squeezing parameter r
num_N = 150 # Number of cooling atoms N_c

ind_r_cut = 100 # Index of r (r ≈ 8.7 dB) at which the squeezing spectrum S(ν) is saved (top of panel c)

# Rb87 level structure (B in G; angular frequencies, with the frequency in GHz)
B = 2.7
gamma = 2*pi*5.75*1e-3 # D1 linewidth
gamma_D2 = 2*pi*6.06*1e-3 # D2 linewidth
w_t = 2*pi*377107.463380 # D1 line
w_t_D2 = 2*pi*384230.48446 # D2 line
w_e = 2*pi*(6.834-3*0.7*1e-3*B)
w_s = w_t+2*pi*0.30544+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*2*0.23*1e-3*B
w_r = w_t-2*pi*0.50906+2*pi*4.271677-2*pi*0.7*1e-3*B+2*pi*0.23*1e-3*B
w_u = w_t_D2-2*pi*0.22985+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*0.93*1e-3*B

N_atom = logrange(100,2e4,num_N) # Number of cooling atoms N_c

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = 0.07*2*pi*1e-3
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = LinRange(0.01,1.5,num_r) # Squeezing parameter
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s).+eps_cav

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and the effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$. Arrays have shape `(num_r, num_N)`.

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav.+3*w_r.-w_s).-sqrt.(4*gr^2*N_atom'.+(w_cav.-w_r).^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom'.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom'*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)'./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

# Step 6: eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom'.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # Coefficient of the s's term

# Step 7: eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom').^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom')./((delta_lamb.^2).+(N_atom'.*dec/2).^2)

## Effective model

Analytical prediction for $\Delta\hat X$ (normalised to the vacuum value).

In [ ]:
aux = (kappa.+dec_f).^2 .+ (kappa.+dec_f).*dec_f.*(-1 .+ exp.(-2*r)) + (2*delta_lamb_f.^2 ./ (kappa.+dec_f)).*(kappa*(1 .+ exp.(-4*r)) .+ 2*dec_f.*exp.(-2*r))
effective = sqrt.(aux./((kappa.+dec_f).^2 .+ 4*delta_lamb_f.^2))

## Cumulant expansion of the full model

In [ ]:
@rnumbers Δ_cav Δ_cav_b Δ_0 Δ_0_b g_b λ_r λ_s χ_b κ κ_b R N α β δ ξ αb βb # Parameters

h_cav = FockSpace(:cavity) # Hilbert space of cavity a
h_atom = NLevelSpace(:atom,(:g,:e)) # Hilbert space of one atom
h_cav_b = FockSpace(:cavity_b) # Hilbert space of cavity b
h = h_cav⊗h_atom⊗h_cav_b # Complete Hilbert space

a = Destroy(h,:a,1)
b = Destroy(h,:b,3)
σ(x,y,k) = IndexedOperator(Transition(h, :σ, x, y), k)

i = Index(h, :i, N, h_atom)

H = Δ_cav*a'*a + Δ_cav_b*b'*b + ∑((Δ_0+Δ_0_b)*σ(:e,:g,i)*σ(:g,:e,i) + λ_r*(a*σ(:e,:g,i)+a'*σ(:g,:e,i)) + λ_s*(a'*σ(:e,:g,i)+a*σ(:g,:e,i)) + g_b*(b*σ(:e,:g,i)+b'*σ(:g,:e,i)) + χ_b*σ(:e,:g,i)*σ(:g,:e,i)*b'*b,i) # Hamiltonian

# Effective spontaneous emission from the eliminated excited levels r, s (D1) and u (D2)
coll_g_eff_r = (α*σ(:g,:e,i)+β*a*σ(:g,:g,i))*sqrt(gamma*dipole_me(1,1,-1,0)^2)
coll_e_eff_r = (α*σ(:e,:e,i)+β*a*σ(:e,:g,i))*sqrt(gamma*dipole_me(2,1,-2,+1)^2)
coll_g_eff_s = (δ*σ(:g,:g,i)+ξ*a*σ(:g,:e,i))*sqrt(gamma*dipole_me(1,2,-1,-1)^2)
coll_e_eff_s = (δ*σ(:e,:g,i)+ξ*a*σ(:e,:e,i))*sqrt(gamma*dipole_me(2,2,-2,0)^2)
coll_g_eff_u = (αb*σ(:g,:e,i)+βb*b*σ(:g,:g,i))*sqrt(gamma_D2*5/12)
coll_e_eff_u = (αb*σ(:e,:e,i)+βb*b*σ(:e,:g,i))*sqrt(gamma_D2/10)

D = [a;b;coll_g_eff_r;coll_e_eff_r;coll_g_eff_s;coll_e_eff_s;coll_g_eff_u;coll_e_eff_u] # Collapse operators
rates = [κ; κ_b; 1; 1; 1; 1; 1; 1]

In [ ]:
ops = [a] # Operators whose equations we derive
eqs = meanfield(ops,H,D;rates=rates,order=2) # Second-order cumulant expansion
me_comp = complete!(eqs) # Complete the set of equations
eqs_sc = scale(me_comp) # Use the permutation symmetry of the atoms

# Two-time correlation functions for the squeezing spectrum
corr = CorrelationFunction(a',a,me_comp; steady_state=true)
corr2 = CorrelationFunction(a,a,me_comp; steady_state=true)
corr_sc = scale(corr)
corr_sc2 = scale(corr2)

@named sys = System(eqs_sc)

In [ ]:
param_list = (Δ_cav, Δ_cav_b, Δ_0, Δ_0_b, g_b, λ_r, λ_s, χ_b, κ, κ_b, R, N, α, β, δ, ξ, αb, βb)
S = Spectrum(corr_sc,param_list)
S2 = Spectrum(corr_sc2,param_list)

## Steady state on the $(r, N_c)$ grid

For each $r$ we first integrate up to the steady state at the largest $N_c$, and then sweep $N_c$ using the previous solution as initial condition. Each point is re-integrated (up to 15 times) until $\langle a^\dagger a\rangle$ varies by less than 1 over the integration window; observables are time-averaged over the last window.

In [ ]:
sq = zeros(num_r,num_N) # ΔX² (normalised to vacuum)
n_s = zeros(num_r,num_N) # Photon number in the squeezed frame
other = zeros(num_r,num_N) # Excited-state population of one atom
out = zeros(num_r,num_N) # Minimum of the squeezing spectrum

freqs = LinRange(-1,1,1000)*(2pi*1e-3)
spec = zeros(num_r,num_N,length(freqs)) # Squeezing spectrum

it = zeros(num_r,num_N)
dist = zeros(num_r,num_N)

progress = Progress(num_r*num_N)
sim_lock = ReentrantLock()

Threads.@threads for ind_r in 1:num_r

    # Initial steady state at the largest N_c
    u0 = zeros(ComplexF64, length(eqs_sc))

    ind_N = num_N

    p0 = (w_c[ind_N], w_c_b[ind_N], w_0[ind_r,ind_N], w_0_b[ind_N], g[ind_N], lambda_r[ind_r,ind_N], lambda_s[ind_r,ind_N], chi_b[ind_N], kappa, kappa_b, r[ind_r], N_atom[ind_N], alfa[ind_r,ind_N], beta[ind_N], delta[ind_r,ind_N], xi[ind_N], alfa_b[ind_N], beta_b[ind_N])
    dict = merge(Dict(unknowns(sys) .=> u0), Dict(param_list .=> p0))

    prob = ODEProblem(sys,dict,(0.0,100000.0))
    sol = solve(prob, Tsit5(); abstol=1e-6, reltol=1e-6, maxiters=1e6)

    u0 = sol.u[end]

    # Sweep over N_c
    for ind_N in 1:num_N

        dist[ind_r,ind_N] = 10

        while dist[ind_r,ind_N]>1 && it[ind_r,ind_N]<15

            p0 = (w_c[ind_N], w_c_b[ind_N], w_0[ind_r,ind_N], w_0_b[ind_N], g[ind_N], lambda_r[ind_r,ind_N], lambda_s[ind_r,ind_N], chi_b[ind_N], kappa, kappa_b, r[ind_r], N_atom[ind_N], alfa[ind_r,ind_N], beta[ind_N], delta[ind_r,ind_N], xi[ind_N], alfa_b[ind_N], beta_b[ind_N])
            dict = merge(Dict(unknowns(sys) .=> u0), Dict(param_list .=> p0))

            prob = ODEProblem(sys,dict,(0.0,10000.0))
            sol = solve(prob, Tsit5(); abstol=1e-6, reltol=1e-6, maxiters=1e5)

            lock(sim_lock) do
                dist[ind_r,ind_N] = maximum(abs.(sol[a'a]))-minimum(abs.(sol[a'a]))
            end

            u0 = sol.u[end]

            it[ind_r,ind_N] = it[ind_r,ind_N]+1

        end

        n_t = length(sol.t)
        u_avg = sum(sol.u)/n_t # Time-averaged state

        spec[ind_r,ind_N,:] = real.(1 .+ kappa*(S(freqs,u_avg,p0) .+ S(-freqs,u_avg,p0) .+ S2(freqs,u_avg,p0) .+ S2(-freqs,u_avg,p0)))

        lock(sim_lock) do
            sq[ind_r,ind_N] = sum(real(sol[a*a]).-real(sol[a]).^2 .+real(sol[a'*a']).-real(sol[a']).^2 .+2*real(sol[a'*a]).+1 .-2*real(sol[a]).*real(sol[a']))/n_t
            other[ind_r,ind_N] = sum(real(sol[σ(:e,:e,1)]))/n_t
            n_s[ind_r,ind_N] = sum(real(sol[a'*a])*cosh(2*r[ind_r]) .+ sinh(r[ind_r])^2 .+ sinh(r[ind_r])*cosh(r[ind_r])*real(sol[a*a]+sol[a'a']))/n_t
            out[ind_r,ind_N] = minimum(spec[ind_r,ind_N,:])
        end

        next!(progress)
    end
end

## Validity conditions and spectra in dB

In [ ]:
cond_r = dec_f./(kappa.*(exp.(2*r).-1)) # >> 1
cond_chi = -other.*chi_b.*N_atom'./w_0_b # << 1
cond_g0 = sqrt.(4*w_0_b.^2 .+(dec.*N_atom').^2)./sqrt.(4*g_02.*N_atom'.*(n_s.+1)) # >> 1

# Stored as -10 log10(S): positive values mean squeezing
Ssq_min = -20*log10.(sqrt.(out))
Ssq_cut = -20*log10.(sqrt.(spec[ind_r_cut,:,:]))

## Save

In [ ]:
data_dir = joinpath("..", "data") # Relative to this notebook

npzwrite(joinpath(data_dir, "N_c.npy"), collect(N_atom))
npzwrite(joinpath(data_dir, "r.npy"), collect(r))
npzwrite(joinpath(data_dir, "freqs_MHz.npy"), collect(freqs/(2pi*1e-3)))
npzwrite(joinpath(data_dir, "dX_full.npy"), sqrt.(sq))
npzwrite(joinpath(data_dir, "dX_eff.npy"), effective)
npzwrite(joinpath(data_dir, "Ssq_min_full.npy"), Ssq_min)
npzwrite(joinpath(data_dir, "Ssq_cut_full.npy"), Ssq_cut)
npzwrite(joinpath(data_dir, "cond_r.npy"), cond_r)
npzwrite(joinpath(data_dir, "cond_chi.npy"), cond_chi)
npzwrite(joinpath(data_dir, "cond_g0.npy"), cond_g0)